In [35]:
!pip install datasets
!pip install transformers
!pip install optuna

In [53]:
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score
from collections import Counter
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

def load_and_prepare_data(filepath='merged_papers.csv',
                        sections_to_include=None,
                        include_abstract=False,
                        include_title=False):
    """Load and preprocess data with robust type handling"""
    # Load data and ensure string type conversion
    df = pd.read_csv(filepath)

    # Initialize text columns list
    text_columns = []

    # 1. Handle sections
    if sections_to_include:
        df = df[df['Category'].isin(sections_to_include)]
        # Convert to string and clean
        df['Section Content'] = df['Section Content'].fillna('').astype(str)
        df = df[df['Section Content'].str.strip() != '']
        text_columns.append('Section Content')

    # 2. Handle abstract
    if include_abstract:
        df['abstract'] = df['abstract'].fillna('').astype(str)
        df = df[df['abstract'].str.strip() != '']
        text_columns.append('abstract')

    # 3. Handle title
    if include_title:
        df['title'] = df['title'].fillna('').astype(str)
        df = df[df['title'].str.strip() != '']
        text_columns.append('title')

    # 4. Combine text sources
    def combine_text(row):
        parts = [str(row[col]).strip() for col in text_columns
                if pd.notna(row[col]) and str(row[col]).strip()]
        return ' '.join(parts) if parts else np.nan

    df['combined_text'] = df.apply(combine_text, axis=1)

    # 5. Final cleaning and label processing
    df = df.dropna(subset=['combined_text', 'decision'])
    df = df[df['combined_text'] != '']
    df['Label'] = pd.factorize(df['decision'])[0]

    print(f"\nFinal dataset size: {len(df)}")
    print("Label distribution:", dict(Counter(df['Label'])))

    return Dataset.from_pandas(df)

def initialize_tokenizer(model_name="allenai/scibert_scivocab_uncased"):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    def tokenize_function(examples):
        return tokenizer(
            examples["combined_text"],
            padding="max_length",
            truncation=True,
            max_length=256
        )
    return tokenize_function

def compute_metrics(p):
    preds = p.predictions.argmax(-1)
    return {
        'accuracy': accuracy_score(p.label_ids, preds),
        'f1': f1_score(p.label_ids, preds, average='weighted')
    }

def main(include_abstract=False, include_title=False, sections_to_include=None):
    # Optimized parameters
    params = {
        "learning_rate": 1.0370844668954537e-05,
        "num_train_epochs": 10,
        "per_device_train_batch_size": 8,
        "per_device_eval_batch_size": 8,
        "weight_decay": 2.0013420622879995e-05,
    }

    # Load data
    dataset = load_and_prepare_data(
        sections_to_include=sections_to_include,
        include_abstract=include_abstract,
        include_title=include_title
    )

    tokenize_function = initialize_tokenizer()
    kf = KFold(n_splits=2, shuffle=True, random_state=42)

    final_metrics = {'accuracy': [], 'f1': [], 'loss': []}
    texts = dataset["combined_text"]
    labels = dataset["Label"]

    for fold, (train_idx, val_idx) in enumerate(kf.split(texts, labels)):
        print(f"\n=== Fold {fold + 1} ===")

        train_data = Dataset.from_dict({
            "combined_text": [texts[i] for i in train_idx],
            "label": [labels[i] for i in train_idx]
        })
        val_data = Dataset.from_dict({
            "combined_text": [texts[i] for i in val_idx],
            "label": [labels[i] for i in val_idx]
        })

        tokenized_train = train_data.map(tokenize_function, batched=True)
        tokenized_val = val_data.map(tokenize_function, batched=True)

        model = AutoModelForSequenceClassification.from_pretrained(
            "allenai/scibert_scivocab_uncased",
            num_labels=len(np.unique(labels)),
            problem_type="single_label_classification"
        )

        training_args = TrainingArguments(
            output_dir=f"./results_fold_{fold+1}",
            eval_strategy="epoch",
            save_strategy="epoch",
            load_best_model_at_end=True,
            metric_for_best_model='f1',
            greater_is_better=True,
            fp16=torch.cuda.is_available(),
            report_to="none",
            **params
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=tokenized_train,
            eval_dataset=tokenized_val,
            compute_metrics=compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
        )

        print("\nTraining...")
        trainer.train()

        print("\nEvaluating...")
        results = trainer.evaluate()

        final_metrics['accuracy'].append(results['eval_accuracy'])
        final_metrics['f1'].append(results['eval_f1'])
        final_metrics['loss'].append(results['eval_loss'])

        print(f"\nFold {fold+1} Results:")
        print(f"Accuracy: {results['eval_accuracy']:.4f}")
        print(f"F1-Score: {results['eval_f1']:.4f}")
        print(f"Loss: {results['eval_loss']:.4f}")

    print("\n=== Final Cross-Validation Results ===")
    print(f"Average Accuracy: {np.mean(final_metrics['accuracy']):.4f}")
    print(f"Average F1-Score: {np.mean(final_metrics['f1']):.4f}")
    print(f"Average Loss: {np.mean(final_metrics['loss']):.4f}")
    print(f"Text sources: Abstract={include_abstract}, Title={include_title}")
    if sections_to_include:
        print(f"Sections included: {sections_to_include}")

In [54]:
import optuna
import numpy as np
from sklearn.model_selection import KFold
from datasets import Dataset

def optimize_hyperparameters(sections_to_include=None,
                           include_abstract=False,
                           include_title=False,
                           n_trials=10):
    """
    Optimize hyperparameters using Optuna

    Args:
        sections_to_include: List of sections to include (e.g., ['Introduction', 'Methods'])
        include_abstract: Whether to include abstract text
        include_title: Whether to include title text
        n_trials: Number of optimization trials

    Returns:
        Dictionary of best hyperparameters found
    """
    def objective(trial):
        # Suggest hyperparameters
        params = {
            "learning_rate": trial.suggest_float("learning_rate", 1e-6, 5e-5, log=True),
            "num_train_epochs": trial.suggest_int("num_train_epochs", 3, 10),
            "per_device_train_batch_size": trial.suggest_categorical("batch_size", [4, 8, 16]),
            "per_device_eval_batch_size": 8,  # Fixed eval batch size
            "weight_decay": trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True),
        }

        # Load data with current settings
        dataset = load_and_prepare_data(
            sections_to_include=sections_to_include,
            include_abstract=include_abstract,
            include_title=include_title
        )

        # Initialize tokenizer
        tokenize_function = initialize_tokenizer()

        # Use 2-fold CV for optimization
        kf = KFold(n_splits=2, shuffle=True, random_state=42)
        texts = dataset["combined_text"]
        labels = dataset["Label"]
        fold_metrics = []

        for fold, (train_idx, val_idx) in enumerate(kf.split(texts, labels)):
            # Prepare datasets
            train_data = Dataset.from_dict({
                "combined_text": [texts[i] for i in train_idx],
                "label": [labels[i] for i in train_idx]
            })
            val_data = Dataset.from_dict({
                "combined_text": [texts[i] for i in val_idx],
                "label": [labels[i] for i in val_idx]
            })

            # Tokenize
            tokenized_train = train_data.map(tokenize_function, batched=True)
            tokenized_val = val_data.map(tokenize_function, batched=True)

            # Initialize model
            model = AutoModelForSequenceClassification.from_pretrained(
                "allenai/scibert_scivocab_uncased",
                num_labels=len(np.unique(labels)),
                problem_type="single_label_classification"
            )

            # Training arguments
            training_args = TrainingArguments(
                output_dir=f"./optuna_trial_{trial.number}_fold_{fold}",
                eval_strategy="epoch",
                save_strategy="epoch",
                load_best_model_at_end=True,
                metric_for_best_model='f1',
                greater_is_better=True,
                fp16=torch.cuda.is_available(),
                report_to="none",
                **params
            )

            trainer = Trainer(
                model=model,
                args=training_args,
                train_dataset=tokenized_train,
                eval_dataset=tokenized_val,
                compute_metrics=compute_metrics,
                callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])

            trainer.train()
            results = trainer.evaluate()
            fold_metrics.append(results['eval_f1'])

        return np.mean(fold_metrics)  # Return average F1 across folds

    # Run optimization
    study = optuna.create_study(direction="maximize")
    study.optimize(objective, n_trials=n_trials)

    # Print results
    print("\n=== Optimization Results ===")
    print(f"Best F1-score: {study.best_value:.4f}")
    print("Best parameters:")
    for key, value in study.best_params.items():
        print(f"  {key}: {value}")

    return study.best_params

In [55]:
from transformers import Trainer
import torch

class CustomTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """
        Handles additional kwargs like num_items_in_batch while maintaining
        the core loss computation functionality
        """
        labels = inputs.get("labels")
        if labels is None:
            raise ValueError("Labels not found in batch inputs")

        outputs = model(**inputs)
        logits = outputs.logits

        # Move labels to same device as logits
        labels = labels.to(logits.device)

        # Handle potential extra dimensions
        if labels.dim() > 1:
            labels = labels.squeeze(-1)

        loss_fct = torch.nn.CrossEntropyLoss()
        loss = loss_fct(
            logits.view(-1, self.model.config.num_labels),
            labels.view(-1)
        )

        return (loss, outputs) if return_outputs else loss

def main(include_abstract=False, include_title=False, sections_to_include=None, params=None):
    """
    Complete working implementation with proper kwargs handling
    """
    # Default hyperparameters
    if params is None:
        params = {
            "learning_rate": 2e-5,
            "num_train_epochs": 3,
            "per_device_train_batch_size": 8,
            "per_device_eval_batch_size": 8,
            "weight_decay": 0.01,
        }

    # Load and prepare data
    dataset = load_and_prepare_data(
        include_abstract=include_abstract,
        include_title=include_title,
        sections_to_include=sections_to_include
    )

    # Tokenization
    tokenizer = AutoTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")

    def tokenize_function(examples):
        tokenized = tokenizer(
            examples["combined_text"],
            padding="max_length",
            truncation=True,
            max_length=256
        )
        tokenized["labels"] = examples["Label"]
        return tokenized

    tokenized_dataset = dataset.map(tokenize_function, batched=True)
    split_dataset = tokenized_dataset.train_test_split(test_size=0.2, seed=42)

    # Model initialization
    model = AutoModelForSequenceClassification.from_pretrained(
        "allenai/scibert_scivocab_uncased",
        num_labels=len(np.unique(dataset["Label"])),
        problem_type="single_label_classification"
    )

    # Training arguments
    training_args = TrainingArguments(
        output_dir="./results",
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        fp16=torch.cuda.is_available(),
        report_to="none",
        **params
    )

    # Training
    trainer = CustomTrainer(
        model=model,
        args=training_args,
        train_dataset=split_dataset["train"],
        eval_dataset=split_dataset["test"],
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
    )

    trainer.train()
    results = trainer.evaluate()

    print("\n=== Final Results ===")
    print(f"Accuracy: {results['eval_accuracy']:.4f}")
    print(f"F1-Score: {results['eval_f1']:.4f}")
    print(f"Loss: {results['eval_loss']:.4f}")

    return results

In [57]:
if __name__ == "__main__":
    # Example configurations:

    # 1. Minimal - title only
    # main(include_title=True)

    # 2. Abstract + title
    # main(include_abstract=True, include_title=True)

    # 3. Specific sections only
    # main(sections_to_include=['introduction', 'methodology'])

    # 4. Full optimization
    best_params = optimize_hyperparameters(
        include_abstract=True,
        include_title=True,
        n_trials=5
    )
    # main(include_abstract=True, include_title=True, params=best_params)

[I 2025-05-19 17:32:32,560] A new study created in memory with name: no-name-2f2a4839-4e6e-4e81-bdad-4272d7cb9c01



Final dataset size: 16405
Label distribution: {0: 11432, 1: 4973}


Map:   0%|          | 0/8202 [00:00<?, ? examples/s]

Map:   0%|          | 0/8203 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at allenai/scibert_scivocab_uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss


[W 2025-05-19 17:34:25,593] Trial 0 failed with parameters: {'learning_rate': 7.648321249156507e-06, 'num_train_epochs': 6, 'batch_size': 16, 'weight_decay': 7.372060052237456e-06} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/optuna/study/_optimize.py", line 197, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "<ipython-input-54-2f367c9c5676>", line 91, in objective
    trainer.train()
  File "/usr/local/lib/python3.11/dist-packages/transformers/trainer.py", line 2245, in train
    return inner_training_loop(
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/transformers/trainer.py", line 2560, in _inner_training_loop
    tr_loss_step = self.training_step(model, inputs, num_items_in_batch)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/transformers/train

KeyboardInterrupt: 